# AHP Priority Ranking - Sample Development Pipeline

## Decision Support Stage

This notebook documents the AHP calculation pipeline for SentiRank. Phase 10C uses sample development judgements only to verify that the backend AHP service, CSV templates, exported metrics, and figures work end to end.

**Important:** outputs in this notebook are marked `sample_development_only` and `not_final_expert_judgement`. They are not final AHP weights and must not be used as thesis ranking results.

## AHP Role in SentiRank

AHP converts expert pairwise judgements into priority weights for the five selected improvement criteria. The criteria come from the selected SVM `merged_5class` aspect taxonomy, but final weighting still requires real expert judgement.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "ml-service").exists()
            and (candidate / "datasets").exists()
            and (candidate / "docs").exists()
        ):
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
TEMPLATE_DIR = PROJECT_ROOT / "docs" / "templates" / "ahp"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ML_SERVICE_DIR:", ML_SERVICE_DIR)

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print(f"Missing JSON: {path}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path):
    if not path.exists():
        print(f"Missing CSV: {path}")
        return None
    return pd.read_csv(path)


def display_image(path: Path) -> None:
    if not path.exists():
        print(f"Missing figure: {path}")
        return
    display(Image(filename=str(path)))


def run_ml_script(script_name: str) -> None:
    command = [sys.executable, "scripts/" + script_name]
    result = subprocess.run(
        command,
        cwd=ML_SERVICE_DIR,
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Script failed: {script_name}")

## Final Candidate Criteria

The `General` fallback label is excluded. These criteria remain expert-validation candidates until real respondents provide pairwise comparisons.

In [ ]:
criteria_payload = load_json(TEMPLATE_DIR / "final_criteria_for_ahp.json")
if criteria_payload:
    criteria_df = pd.DataFrame(criteria_payload["criteria"])
    display(criteria_df[["criterion_id", "criterion_name", "description", "source_labels", "expert_validation_required"]])

## Sample AHP Input

The sample input contains exactly 10 pairwise comparisons for 5 criteria. It is deliberately consistent for pipeline validation and is not a substitute for expert judgement.

In [ ]:
ahp_sample_input = TEMPLATE_DIR / "sample_development" / "ahp_pairwise_sample_development.csv"
ahp_input_df = load_csv(ahp_sample_input)
if ahp_input_df is not None:
    display(ahp_input_df)
    print("Pairwise comparisons:", len(ahp_input_df))

## Run Sample AHP Calculation

This cell runs only the sample-development script. It writes to `datasets/outputs/eda/06_ahp/sample_development/` and does not create final `ahp_weights.csv` or ranking outputs.

In [ ]:
run_ml_script("calculate_ahp_from_expert_judgement.py")

## Sample AHP Outputs

The outputs below are useful for checking the pipeline shape: weights, pairwise matrix, and consistency ratio.

In [ ]:
AHP_OUTPUT_DIR = PROJECT_ROOT / "datasets" / "outputs" / "eda" / "06_ahp" / "sample_development"
AHP_FIGURE_DIR = PROJECT_ROOT / "docs" / "figures" / "06_ahp" / "sample_development"

weights_df = load_csv(AHP_OUTPUT_DIR / "ahp_weights_sample_development.csv")
if weights_df is not None:
    display(weights_df.sort_values("rank"))

consistency = load_json(AHP_OUTPUT_DIR / "ahp_consistency_sample_development.json")
if consistency:
    display(pd.DataFrame([consistency]))

matrix_df = load_csv(AHP_OUTPUT_DIR / "ahp_pairwise_matrix_sample_development.csv")
if matrix_df is not None:
    display(matrix_df)

## Sample AHP Weight Figure

In [ ]:
display_image(AHP_FIGURE_DIR / "ahp_weights_sample_development.png")

## Interpretation Notes

- The expected sample CR is approximately `0` because the development matrix was constructed to be internally consistent.
- Real expert responses must still be checked against the `CR <= 0.10` threshold.
- Final AHP weights and priority rankings are intentionally not generated in Phase 10C.